# Baselines

> Probably better than the other models

In [ ]:
#| default_exp baselines

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, math, torch.nn as nn, torch.nn.functional as F, lightning.pytorch as pl
from physiojepa.layers import Patch, InceptionBlock
from physiojepa.tokenizers import PatchEncoder


from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

## FCN

In [ ]:
#| export
class FCN(nn.Module):
    def __init__(self, 
                 c_in,
                 layers=[128, 256, 128], # kwargs for the encoder layer
                 kernel_sizes=[7, 5, 3], # kwargs for the encoder layer
                 n_classes=1, # kwargs for the encoder layer
                 ):
        super().__init__()
        self.convblocks = nn.ModuleList()
        for i in range(len(layers)):
            self.convblocks.append(nn.Sequential(
                nn.Conv1d(c_in if i == 0 else layers[i-1], layers[i], kernel_size=kernel_sizes[i], bias=False, padding='same'),
                nn.BatchNorm1d(layers[i]),
                nn.ReLU()
            ))
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(layers[-1], n_classes)

    def forward(self, x):
        """
        In: [bs, n_channels, seq_len]
        """
        for convblock in self.convblocks:
            x = convblock(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

## InceptionTime

This is the single-network InceptionTime classifier flow used by the established tsai PyTorch implementation: a six-module Inception block, global average pooling, and a linear classification head. The original InceptionTime paper ensembles five independently initialized copies of this network.

In [ ]:
#| export
class InceptionTime(nn.Module):
    """Single InceptionTime network with the reference block-pool-linear flow."""
    def __init__(self,
                 c_in,
                 c_out=1,
                 seq_len=None,
                 nf=32,
                 nb_filters=None,
                 residual=True,
                 depth=6,
                 kernel_size=40,
                 bottleneck=True,
                 ):
        super().__init__()
        nf = nf if nf is not None else nb_filters
        if nf is None:
            raise ValueError("nf or nb_filters must specify the number of Inception filters")
        self.seq_len = seq_len
        self.inceptionblock = InceptionBlock(
            in_channels=c_in,
            bottleneck_channels=nf,
            residual=residual,
            depth=depth,
            groups=1,
            kernel_size=kernel_size,
            bottleneck=bottleneck,
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(nf * 4, c_out)

    def forward(self, x):
        """In: ``[batch, channels, time]``; out: ``[batch, c_out]`` logits."""
        x = self.inceptionblock(x)
        x = self.gap(x)
        x = self.flatten(x)
        return self.fc(x)

In [ ]:
# Reference-flow regression checks.
_inceptiontime = InceptionTime(c_in=3, c_out=1)
assert len(_inceptiontime.inceptionblock.inception) == 6
assert len(_inceptiontime.inceptionblock.shortcut) == 2
assert [conv.kernel_size[0] for conv in _inceptiontime.inceptionblock.inception[0].convs] == [39, 19, 9]
assert sum(p.numel() for p in _inceptiontime.parameters()) == 455_361
assert not any(isinstance(module, (nn.Sigmoid, nn.Softmax)) for module in _inceptiontime.modules())
_inceptiontime_x = torch.randn(2, 3, 257)
_inceptiontime_features = _inceptiontime.inceptionblock(_inceptiontime_x)
assert _inceptiontime_features.shape == (2, 128, 257)
assert _inceptiontime(_inceptiontime_x).shape == (2, 1)

## General Baseline

### Lightning

In [ ]:
#| export
class GeneralTimeSupervised(pl.LightningModule):
    def __init__(self, 
                 supervised_model, # kwargs for the encoder layer
                 learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                 train_size, # the training data size (for one_cycle_scheduler=True)
                 batch_size,
                 n_gpus,
                 n_classes = 1,
                 n_labels = 1,
                 metrics={}, # name:function for metrics to log
                 loss_fxn='CrossEntropy', # loss function to use, can be CrossEntropy
                 gamma=2., # for focal loss
                 class_weights=None, # weights of classes to use in CE loss fxn
                 label_smoothing=0, # label smoothing for cross entropy loss
                 y_padding_mask=-100, # padded value that was added to target and indice to ignore when computing loss
                 epochs=100, # number of epochs for one_cycle_scheduler
                 optimizer_type='AdamW',
                 scheduler_type='OneCycle',
                 weight_decay=0., # weight decay for Adam optimizer
                 final_weight_decay=0.4, # final weight decay for weight decay scheduler
                 use_weight_decay_scheduler=False, # whether to use a weight decay scheduler
                 transforms=None, # transforms to apply to the data
                 mixup_callback=None, # mixup callback to apply to the data
                 scheduler_kwargs={},
                 ):
        super().__init__()
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.label_smoothing = label_smoothing
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size // self.batch_size
        self.y_padding_mask = y_padding_mask
        self.class_weights = class_weights
        self.loss_fxn = loss_fxn
        self.n_labels = n_labels
        self.n_classes = n_classes
        self.metrics = nn.ModuleDict(metrics)
        assert not (self.n_classes > 1 and self.n_labels > 1), "Cannot have both n_classes > 1 and n_labels > 1"
        self.gamma = gamma
        self.optimizer_type = optimizer_type
        self.scheduler_kwargs = scheduler_kwargs
        self.model = supervised_model
        self.transforms = transforms
        self.mixup_callback = mixup_callback
        self.save_hyperparameters()

    def forward(self, x):
        """
        In: [bs, n_channels, seq_len]
        """
        x = self.model(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        return x
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x,y = batch
        preds = self.model(x)
        return preds,y
        
    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        if self.mixup_callback is not None:
            batch = self.mixup_callback(batch)
        x, y = batch
        x = self.model(x)
        if self.n_classes == 1:
            ce_loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
            y = y.float()
        else:
            ce_loss = nn.CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
        
        loss = ce_loss(x,y)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
      
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        x = self.model(x)
        if self.n_classes == 1:
            ce_loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
            y = y.float()
        else:
            ce_loss = nn.CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
        
        loss = ce_loss(x,y)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        #self.log('val_ce_loss' if self.loss_fxn != 'CrossEntropy' else 'val_focal_loss', ce_loss_val if self.loss_fxn != 'CrossEntropy' else focal_loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        if self.n_classes > 1:
            x_probs = torch.softmax(x, dim=1)
        else:
            x_probs = torch.sigmoid(x)
        for metric in self.metrics:
            self.metrics[metric].update(x_probs, y.long())

    def on_validation_epoch_start(self):
        torch.cuda.empty_cache()
    
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def on_validation_epoch_end(self):
        for name, metric in self.metrics.items():
            metric_val = metric.compute()
            if metric_val.dim() != 0:
                self.log_dict({f'val_{name}_{i}':metric_val[i] for i in range(len(metric_val))}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            else:
                self.log(f'val_{name}', metric_val, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
        
    def configure_optimizers(self):
        param_groups = [{'params': (p for n, p in self.model.named_parameters() if ('bias' not in n) and (len(p.shape) != 1))}, 
                        {'params': (p for n, p in self.model.named_parameters() if ('bias' in n) or (len(p.shape) == 1)), 'WD_exclude': True, 'weight_decay': 0}]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False) if self.optimizer_type == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, epochs=self.epochs, steps_per_epoch=self.ipe, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()